In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import pandas as pd
import csv
import time

# ---------------- CONFIG ----------------
INPUT_FILE = "facebook_followers.csv"   # Input CSV with column 'Profile Link'
OUTPUT_FILE = "facebook_places_lived.csv"  # Output file

chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("--disable-notifications")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("detach", True)

driver = webdriver.Chrome(options=chrome_options)

# ---------------- LOGIN ----------------
driver.get("https://www.facebook.com/")
print("🔐 Please log in to Facebook...")
input("👉 Press Enter after you have logged in: ")

# ---------------- READ INPUT ----------------
df = pd.read_csv(INPUT_FILE)
if "Profile Link" not in df.columns:
    raise Exception("⚠ CSV must contain a column named 'Profile Link'.")
profile_links = df["Profile Link"].dropna().unique().tolist()
print(f"✅ Loaded {len(profile_links)} profile URLs.\n")

# ---------------- PREPARE OUTPUT ----------------
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Profile URL", "All Places Lived"])


# ---------------- SCRAPER FUNCTION ----------------
def scrape_places(url):
    print(f"➡ Extracting Places Lived from: {url}")

    if "?id=" in url:
        about_url = url + "&sk=about"
    else:
        about_url = url + "?sk=about"

    driver.get(about_url)
    time.sleep(5)

    # Try clicking “Places lived”
    try:
        selectors = [
            "//span[text()='Places lived']",
            "//a[contains(text(),'Places lived')]",
            "//div[contains(text(),'Places lived')]"
        ]
        for s in selectors:
            try:
                btn = driver.find_element(By.XPATH, s)
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(3)
                break
            except:
                pass
    except:
        pass

    # Collect ALL possible places
    places = set()

    # Method 1: Look for spans inside Places lived section
    try:
        elems = driver.find_elements(
            By.XPATH,
            "//div[contains(@aria-label,'Places lived')]//span[@dir='auto']"
        )
        for e in elems:
            t = e.text.strip()
            if t and len(t) > 2 and t.lower() != "places lived":
                places.add(t)
    except:
        pass

    # Method 2: Backup — any span that looks like a location
    try:
        all_spans = driver.find_elements(By.XPATH, "//span[@dir='auto']")
        for s in all_spans:
            t = s.text.strip()
            if "," in t and len(t) <= 60:  # simple filter
                places.add(t)
    except:
        pass

    places = list(places)

    print("   ✓ Found Places:", places)

    return " | ".join(places)


# ---------------- MAIN LOOP ----------------
for i, url in enumerate(profile_links, start=1):
    print(f"\n📄 [{i}/{len(profile_links)}] Scraping {url}")
    all_places = scrape_places(url)

    with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([url, all_places])

    time.sleep(2)

print("\n✅ DONE — Saved to:", OUTPUT_FILE)
driver.quit()

🔐 Please log in to Facebook...


👉 Press Enter after you have logged in:  


✅ Loaded 3235 profile URLs.


📄 [1/3235] Scraping https://www.facebook.com/feverheadproductions
➡ Extracting Places Lived from: https://www.facebook.com/feverheadproductions
   ✓ Found Places: ['Ocala, FL, US']
